# Phase 1: Data Understanding & Profiling
## E-Commerce Sales & Customer Analytics Dashboard

This notebook performs detailed profiling on the raw Olist Brazilian E-Commerce dataset. We will load all 9 operational tables and extract:
- Number of rows and columns
- Missing value counts
- Duplicate records count
- Core data types
- Complete Data Dictionary with business definitions

---

In [ ]:
import pandas as pd
import numpy as np
import os

# Set dataset folder path
RAW_DATA_DIR = '../data/raw/'
print(f'Raw dataset directory: {RAW_DATA_DIR}')
print(os.listdir(RAW_DATA_DIR))

### Loading the Tables
We will load all the CSV files into pandas dataframes to check their schema.

In [ ]:
datasets = {
    'customers': 'olist_customers_dataset.csv',
    'geolocation': 'olist_geolocation_dataset.csv',
    'order_items': 'olist_order_items_dataset.csv',
    'order_payments': 'olist_order_payments_dataset.csv',
    'order_reviews': 'olist_order_reviews_dataset.csv',
    'orders': 'olist_orders_dataset.csv',
    'products': 'olist_products_dataset.csv',
    'sellers': 'olist_sellers_dataset.csv',
    'category_translation': 'product_category_name_translation.csv'
}

dfs = {}
for name, filename in datasets.items():
    filepath = os.path.join(RAW_DATA_DIR, filename)
    dfs[name] = pd.read_csv(filepath)
    print(f'Loaded {name}: {dfs[name].shape[0]} rows, {dfs[name].shape[1]} columns')

### Detailed Profiling for Every Table
Let's write a function to summarize metadata: dimensions, null counts, duplicate records, and data types.

In [ ]:
def profile_table(name, df):
    print('='*50)
    print(f'PROFILE FOR TABLE: {name.upper()}')
    print('='*50)
    print(f'Dimensions: {df.shape[0]} rows, {df.shape[1]} columns\n')
    
    print('--- Data Types and Missing Values ---')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    types = df.dtypes
    
    profile_df = pd.DataFrame({
        'Data Type': types,
        'Missing Values': missing,
        'Missing %': missing_pct.round(2)
    })
    print(profile_df)
    
    duplicates = df.duplicated().sum()
    print(f'\nDuplicate Records: {duplicates} ({duplicates/len(df)*100:.2f}%)\n')
    
    print('--- Sample Rows ---')
    display(df.head(2))
    print('\n')

for name, df in dfs.items():
    profile_table(name, df)

### Complete Data Dictionary

Based on the profiling, here is the business dictionary for key fields:

#### 1. `customers`
- `customer_id`: Unique key assigned to each order transaction (changes for every purchase).
- `customer_unique_id`: Persistent identifier for the physical customer (remains constant across multiple purchases).
- `customer_zip_code_prefix`: Customer's zip code (first 5 digits).
- `customer_city`: Customer's city.
- `customer_state`: Customer's state code (e.g., SP, RJ).

#### 2. `orders`
- `order_id`: Primary key for the transaction.
- `customer_id`: Foreign key linking to `customers` table.
- `order_status`: Lifecycle status (delivered, shipped, canceled, invoiced, processing, approved, created, unavailable).
- `order_purchase_timestamp`: Date and time the order was placed.
- `order_approved_at`: Date and time payment was approved.
- `order_delivered_carrier_date`: Date and time order was handed over to logistics.
- `order_delivered_customer_date`: Date and time customer received the package.
- `order_estimated_delivery_date`: Promised delivery date communicated at checkout.

#### 3. `order_items`
- `order_id`: Foreign key linking to `orders` table.
- `order_item_id`: Sequential line item number within the same order (e.g., 1, 2, 3).
- `product_id`: Foreign key linking to `products` table.
- `seller_id`: Foreign key linking to `sellers` table.
- `shipping_limit_date`: Seller shipping deadline to hand over to carrier.
- `price`: Item price (in BRL).
- `freight_value`: Shipping cost charged to customer (in BRL).

#### 4. `order_payments`
- `order_id`: Foreign key linking to `orders` table.
- `payment_sequential`: Order payment step sequence (if using multiple payment methods).
- `payment_type`: Method of payment (credit_card, boleto, voucher, debit_card, not_defined).
- `payment_installments`: Selected installment count for credit card purchases.
- `payment_value`: Total amount paid (price + freight) for that payment transaction.

#### 5. `order_reviews`
- `review_id`: Unique review identifier.
- `order_id`: Foreign key linking to `orders` table.
- `review_score`: Satisfaction rating from 1 (lowest) to 5 (highest).
- `review_comment_title`: Title of the review comment.
- `review_comment_message`: Text review left by the customer.
- `review_creation_date`: Review survey creation timestamp.
- `review_answer_timestamp`: Review submission timestamp.

#### 6. `products`
- `product_id`: Primary key for the product.
- `product_category_name`: Category name in Portuguese.
- `product_name_lenght`: Character count of the product title.
- `product_description_lenght`: Character count of the product description.
- `product_photos_qty`: Count of photos uploaded for the product.
- `product_weight_g`: Weight of product in grams.
- `product_length_cm`: Product length in centimeters.
- `product_height_cm`: Product height in centimeters.
- `product_width_cm`: Product width in centimeters.

#### 7. `sellers`
- `seller_id`: Primary key for the seller.
- `seller_zip_code_prefix`: Seller's zip code (first 5 digits).
- `seller_city`: Seller's city.
- `seller_state`: Seller's state code.

#### 8. `geolocation`
- `geolocation_zip_code_prefix`: First 5 digits of zip code.
- `geolocation_lat`: Latitude.
- `geolocation_lng`: Longitude.
- `geolocation_city`: City name.
- `geolocation_state`: State code.

#### 9. `product_category_translation`
- `product_category_name`: Portuguese category name.
- `product_category_name_english`: English category translation.
